In [1]:
from pathlib import Path
from zipfile import ZipFile
import json
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go

DATA = Path("data")
DATA.mkdir(exist_ok=True)
TSE = "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2022/"
NAMES = {2018: ("Fernando Haddad", "Jair Bolsonaro"),
         2022: ("Lula", "Jair Bolsonaro"),
         2026: ("Lula", "Flávio Bolsonaro")}
URLS = {
    (2018, "votes"): "https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_secao/votacao_secao_2018_BR.zip",
    (2018, "times"): "https://cdn.tse.jus.br/estatistica/sead/odsele/detalhe_votacao_secao/detalhe_votacao_secao_2018.zip",
    (2026, 1): "https://raw.githubusercontent.com/gfvdata-web/eleicoes-2026/main/docs/data/apuracao/evolucao-presidente.json",
    (2026, 2): "https://raw.githubusercontent.com/joaogallo/projecao-apuracao-eleicoes/main/resultados/dashboard.json",
    **{(2022, r): TSE + f"Historico_Totalizacao_Presidente_BR_{r}T_2022.zip" for r in (1, 2)}
}
EXPECTED = {(2022, 1): (57_259_504, 51_072_345),
            (2022, 2): (60_345_999, 58_206_354),
            (2026, 2): (64_256_232, 59_826_041)}

In [2]:
def download(key):
    url = URLS[key]
    path = DATA / url.rsplit("/", 1)[-1].lower()
    if not path.exists():
        with requests.get(url, stream=True, timeout=90) as response:
            response.raise_for_status()
            with open(path.with_suffix(".part"), "wb") as f:
                for chunk in response.iter_content(1 << 20):
                    f.write(chunk)
        path.with_suffix(".part").replace(path)
        print("Downloaded:", path.name)
    return path


def load_2018(round_):
    cache = DATA / f"replay_2018_{round_}t_v3.pkl"
    if cache.exists():
        df = pd.read_pickle(cache)
        if not df.empty and {"timestamp", "sections", "validos", "votes_a", "votes_b"}.issubset(df):
            return df

    keys = ["NR_TURNO", "SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"]

    def national_csv(z, filename):
        matches = [name for name in z.namelist() if Path(name).name.lower() == filename.lower()]
        if len(matches) != 1:
            raise ValueError(f"Arquivo nacional {filename} ausente/duplicado. CSVs: {z.namelist()[:12]}")
        return matches[0]

    def normalize(chunk):
        chunk["SG_UF"] = chunk["SG_UF"].str.strip().str.upper()
        for col in ("NR_TURNO", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"):
            chunk[col] = pd.to_numeric(chunk[col].str.strip(), errors="coerce").astype("Int32")
        return chunk

    times = []
    observed = set()
    with ZipFile(download((2018, "times"))) as z:
        name = national_csv(z, "detalhe_votacao_secao_2018_BR.csv")
        with z.open(name) as f:
            cols = pd.read_csv(f, sep=";", encoding="latin1", nrows=0).columns.tolist()
        clock = "DT_PRIM_TOT_PARCIAL_HOR_TSE"
        if clock not in cols:
            raise ValueError(f"Coluna {clock} não encontrada: {cols}")
        date_col = "DT_PRIM_TOT_PARCIAL" if "DT_PRIM_TOT_PARCIAL" in cols else None
        fields = keys + ["CD_CARGO", clock] + ([date_col] if date_col else [])
        with z.open(name) as f:
            for chunk in pd.read_csv(f, sep=";", encoding="latin1", dtype="string", usecols=fields,
                                     chunksize=250_000):
                turns = pd.to_numeric(chunk.NR_TURNO.str.strip(), errors="coerce")
                cargos = pd.to_numeric(chunk.CD_CARGO.str.strip(), errors="coerce")
                observed.update((int(t), int(c)) for t, c in
                                pd.DataFrame({"t": turns, "c": cargos}).dropna().drop_duplicates()
                                .itertuples(index=False, name=None))
                chunk = chunk.loc[turns.eq(round_) & cargos.eq(1)].copy()
                if chunk.empty:
                    continue
                chunk = normalize(chunk)
                text = chunk[clock].str.strip()
                stamp = pd.to_datetime(text, dayfirst=True, errors="coerce", format="mixed")
                if date_col:
                    date = chunk[date_col].str.extract(r"(\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2})", expand=False)
                    hour = text.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?)", expand=False)
                    combined = pd.to_datetime(date + " " + hour, dayfirst=True,
                                              errors="coerce", format="mixed")
                    stamp = combined.fillna(stamp)
                chunk["timestamp"] = stamp
                times.append(chunk[keys + ["timestamp"]].dropna())

    if not times:
        raise ValueError(f"2018: nenhum registro presidencial no  {round_}º turno. "
                         f"Códigos turno/cargo encontrados na amostra: {sorted(observed)[:30]}")
    times = pd.concat(times, ignore_index=True).drop_duplicates(keys)
    print(f"2018/{round_}: {len(times):,} seções com horário válido")

    parts = []
    with ZipFile(download((2018, "votes"))) as z:
        name = national_csv(z, "votacao_secao_2018_BR.csv")
        fields = keys + ["CD_CARGO", "NR_VOTAVEL", "QT_VOTOS"]
        with z.open(name) as f:
            for chunk in pd.read_csv(f, sep=";", encoding="latin1", dtype="string", usecols=fields,
                                     chunksize=250_000):
                turns = pd.to_numeric(chunk.NR_TURNO.str.strip(), errors="coerce")
                cargos = pd.to_numeric(chunk.CD_CARGO.str.strip(), errors="coerce")
                numbers = pd.to_numeric(chunk.NR_VOTAVEL.str.strip(), errors="coerce")
                chunk = chunk.loc[turns.eq(round_) & cargos.eq(1) & numbers.between(10, 94)].copy()
                if chunk.empty:
                    continue
                chunk = normalize(chunk)
                chunk["NR_VOTAVEL"] = pd.to_numeric(chunk.NR_VOTAVEL, errors="coerce")
                chunk["QT_VOTOS"] = pd.to_numeric(chunk.QT_VOTOS, errors="coerce").fillna(0).astype("int64")
                chunk["votes_a"] = chunk.QT_VOTOS.where(chunk.NR_VOTAVEL.eq(13), 0)
                chunk["votes_b"] = chunk.QT_VOTOS.where(chunk.NR_VOTAVEL.eq(17), 0)
                parts.append(chunk.groupby(keys, as_index=False)[["QT_VOTOS", "votes_a", "votes_b"]].sum())

    if not parts:
        raise ValueError("2018: não foram encontrados votos presidenciais no arquivo nacional.")
    votes = pd.concat(parts).groupby(keys, as_index=False)[["QT_VOTOS", "votes_a", "votes_b"]].sum()
    joined = votes.merge(times, on=keys, how="inner", validate="one_to_one")
    print(f"2018/{round_}: {len(votes):,} seções com votos; {len(joined):,} seções associadas")
    if joined.empty:
        raise ValueError("2018: votos e horários não coincidem pelas chaves das seções.")

    joined["timestamp"] = joined.timestamp.dt.ceil("min")
    df = joined.groupby("timestamp", as_index=False).agg(
        sections=("NR_SECAO", "size"), validos=("QT_VOTOS", "sum"),
        votes_a=("votes_a", "sum"), votes_b=("votes_b", "sum"))
    df[["sections", "validos", "votes_a", "votes_b"]] = df[["sections", "validos", "votes_a", "votes_b"]].cumsum()
    if df.empty or not df.validos.gt(0).any():
        raise ValueError("2018: reconstrução sem votos válidos; cache não gravado.")
    df.to_pickle(cache)
    return df

def load_counting(year, round_):
    path = download((year, round_)) if year != 2018 else None
    if year == 2022:
        with ZipFile(path) as z:
            csv = next(n for n in z.namelist() if n.lower().endswith(".csv"))
            with z.open(csv) as f:
                raw = pd.read_csv(f, sep=";", encoding="latin1", dtype="string")
        raw.columns = raw.columns.str.strip().str.upper()
        for col in ("SG_UE_UF", "SG_UE", "SG_UF"):
            if col in raw:
                raw = raw.loc[raw[col].str.strip().str.upper().eq("BR")]
                break
        if "CD_CARGO" in raw:
            raw = raw.loc[pd.to_numeric(raw.CD_CARGO, errors="coerce").eq(1)]
        def candidate(token):
            found = [c for c in raw if c.endswith("_QT_VOTOS_TOT_ACUMULADO") and token in c]
            if len(found) != 1: raise ValueError(f"Candidate column {token}: {found}")
            return found[0]
        cols = {"sections": "QT_SECOES_TOT_ACUMULADO", "validos": "QT_VOTOS_CONCORRENTES_ACUMULADO",
                "votes_a": candidate("LULA"), "votes_b": candidate("BOLSONARO")}
        df = pd.DataFrame({k: pd.to_numeric(raw[v], errors="coerce") for k, v in cols.items()})
        df["timestamp"] = pd.to_datetime(raw.DT_TOTALIZACAO, dayfirst=True, format="mixed", errors="coerce")
        df = df.dropna().loc[lambda d: d.validos > 0].sort_values(["timestamp", "sections"]).reset_index(drop=True)
        df["share_a"], df["share_b"] = df.votes_a / df.validos, df.votes_b / df.validos
        df["progress"] = 100 * df.sections / df.sections.max()
        source = "Histórico oficial do TSE"

    elif (year, round_) == (2026, 1):
        raw = json.loads(path.read_text(encoding="utf-8"))
        names = [v[0].upper() for v in raw["candidatos"]]
        a, b = (next(i for i, n in enumerate(names) if token in n) for token in ("LULA", "BOLSONARO"))
        df = pd.DataFrame({"timestamp": [p[0] for p in raw["pontos"]],
                           "progress": [p[1] for p in raw["pontos"]],
                           "share_a": [p[2][a] / 100 for p in raw["pontos"]],
                           "share_b": [p[2][b] / 100 for p in raw["pontos"]]})
        df = df.sort_values("progress").reset_index(drop=True)
        source = "Parciais do TSE arquivadas pelo projeto gfvdata (2026)"

    elif (year, round_) == (2026, 2):
        raw = json.loads(path.read_text(encoding="utf-8"))
        snapshots = next((v.get("oficial") for v in raw["cenarios"] if v.get("turno") == 2), None)
        if not snapshots: raise ValueError("Série oficial do 2º turno ausente do arquivo.")
        df = pd.DataFrame(snapshots).rename(columns={"pct_secoes": "progress", "hora": "timestamp",
                          "secoes": "sections", "validos": "validos",
                          "votos_lula": "votes_a", "votos_bolsonaro": "votes_b"})
        for col in ("sections", "validos", "votes_a", "votes_b"):
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.dropna(subset=["sections", "validos", "votes_a", "votes_b"]).loc[lambda d: d.validos > 0]
        df = df.sort_values("sections").reset_index(drop=True)
        df["share_a"], df["share_b"] = df.votes_a / df.validos, df.votes_b / df.validos
        df["progress"] = 100 * df.sections / df.sections.max()
        source = "Parciais do TSE arquivadas pelo projeto Contagem Direta"

    elif year == 2018:
        df = load_2018(round_)
        df["share_a"], df["share_b"] = df.votes_a / df.validos, df.votes_b / df.validos
        df["progress"] = 100 * df.sections / df.sections.max()
        source = "Reconstrução cronológica de 2018 (método Números da Eleição; arquivos TSE)"

    else:
        raise ValueError("Ano/turno indisponível")

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["progress", "share_a", "share_b"])
    df = df.loc[df.progress.between(0, 100) & df.share_a.between(0, 1) & df.share_b.between(0, 1)]
    df = df.sort_values("progress").reset_index(drop=True)
    if df.empty: raise ValueError("Não há observações válidas.")
    if (year, round_) in EXPECTED:
        got = tuple(df[["votes_a", "votes_b"]].iloc[-1].astype(int))
        if got != EXPECTED[(year, round_)]:
            raise ValueError(f"Totais divergentes: {got} != {EXPECTED[(year, round_)]}")
    if (year, round_) == (2026, 1) and (abs(df.share_a.iloc[-1] - .4516) > .001 or abs(df.share_b.iloc[-1] - .4703) > .001):
        raise ValueError("Parciais de 2026/1 não conferem com resultado final.")

    df.attrs["source"] = source
    return df


def plot_counting(df, year, round_, x_axis="progress", step=None, x_range=None, y_range=None):
    if x_axis not in df.columns: raise ValueError(f"Eixo {x_axis} indisponível; utilize 'progress'.")
    d = df.copy()
    if step is not None:
        checkpoints = np.arange(step, 100 + step / 2, step)
        indices = np.searchsorted(d.progress.to_numpy(), checkpoints, side="right") - 1
        d = d.iloc[np.unique(indices[indices >= 0])]
    if x_range is not None: d = d.loc[d[x_axis].between(*x_range)]
    if d.empty: raise ValueError("Não há dados na faixa selecionada.")
    fig = go.Figure()
    for name, col, color in zip(NAMES[year], ("share_a", "share_b"), ("#d92f35", "#3455d9")):
        fig.add_trace(go.Scattergl(x=d[x_axis], y=d[col], name=name, mode="lines+markers",
                                 line=dict(color=color, width=1.2), marker=dict(color=color, size=4),
                                 hovertemplate="Progresso: %{x:,.2f}<br>Votos válidos: %{y:.4%}<extra>" + name + "</extra>"))
    fig.update_layout(title=f"Eleições presidenciais de {year} — {round_}º turno<br><sup>{df.attrs['source']}</sup>",
                      template="plotly_white", height=620, hovermode="closest", dragmode="zoom",
                      xaxis_title="Seções totalizadas (%)" if x_axis == "progress" else "Seções totalizadas",
                      yaxis_title="Proporção acumulada dos votos válidos")
    fig.update_xaxes(range=x_range, tickformat=".1f" if x_axis == "progress" else "~s")
    fig.update_yaxes(range=y_range, tickformat=".1%")
    fig.show(renderer="vscode")
    return fig

In [ ]:
# Download all relevant datasets
for YEAR, ROUND in [(2018, 1), (2018, 2), (2022, 1), (2022, 2), (2026, 1), (2026, 2)]:

    df = load_counting(YEAR, ROUND)
    print(f"{df.attrs['source']} — {len(df):,} observações")
    display(df.tail(3))

2018/1: 454,490 seções com horário válido
2018/1: 454,446 seções com votos; 454,446 seções associadas
Reconstrução cronológica de 2018 (método Números da Eleição; arquivos TSE) — 417 observações


,timestamp,sections,validos,votes_a,votes_b,share_a,share_b,progress
414,2018-10-08 16:51:00,454444,107050465,31341952,49276882,0.292777,0.460315,99.99956
415,2018-10-08 21:21:00,454445,107050608,31341960,49276975,0.292777,0.460315,99.99978
416,2018-10-23 17:17:00,454446,107050749,31342051,49277010,0.292778,0.460314,100.0


Reconstrução cronológica de 2018 (método Números da Eleição; arquivos TSE) — 287 observações


,timestamp,sections,validos,votes_a,votes_b,share_a,share_b,progress
284,2018-10-29 14:59:00,454438,104837005,47040103,57796902,0.448698,0.551302,99.99802
285,2018-10-29 15:22:00,454439,104837041,47040107,57796934,0.448697,0.551303,99.99824
286,2019-02-25 18:11:00,454447,104838753,47040906,57797847,0.448698,0.551302,100.0


Downloaded: historico_totalizacao_presidente_br_1t_2022.zip
Histórico oficial do TSE — 8,437 observações


...

,sections,validos,votes_a,votes_b,timestamp,share_a,share_b,progress
5283,472073,118551988,60345737,58206251,2022-10-30 22:55:00,0.509023,0.490977,99.999576
5284,472074,118552147,60345825,58206322,2022-10-30 22:57:09,0.509023,0.490977,99.999788
5285,472075,118552353,60345999,58206354,2022-10-31 00:18:04,0.509024,0.490976,100.0


Downloaded: evolucao-presidente.json
Parciais do TSE arquivadas pelo projeto gfvdata (2026) — 48 observações


,timestamp,progress,share_a,share_b
45,22:34,99.92,0.4514,0.4705
46,22:36,99.93,0.4514,0.4705
47,12:51,100.00,0.4516,0.4703


Downloaded: dashboard.json
Parciais do TSE arquivadas pelo projeto Contagem Direta — 367 observações


,progress,timestamp,sections,validos,votes_a,votes_b,parcial_lula,parcial_bolsonaro,share_a,share_b
364,99.9998,23:09,499247,124081971,64255969,59826002,0.51785,0.48215,0.517851,0.482149
365,100.0000,23:22,499248,124082273,64256232,59826041,0.51785,0.48215,0.517852,0.482148
366,100.0000,23:27,499248,124082273,64256232,59826041,0.51785,0.48215,0.517852,0.482148


In [7]:
X_AXIS = "progress"  # "sections" when available
YEAR, ROUND = 2022, 2
df = load_counting(YEAR, ROUND)

fig = plot_counting(df, YEAR, ROUND, x_axis=X_AXIS)